# Interactive Force Predictions Debugging

This notebook provides an interactive way to debug force predictions from the BioEmu model. It allows you to:
1. Visualize a single frame and its predicted forces
2. Override the protein configuration
3. Compare forces with different noise levels
4. Visualize the results using nglview

In [8]:
import hydra
import torch
import numpy as np
from bioemu.datasets.fastfolders import FastFolderTrajectory
from bioemu.interpolate import Interpolator
import mdtraj as md
import nglview as nv
from typing import cast
import matplotlib.pyplot as plt

## Configuration

You can modify these parameters to change the protein and other settings:

In [2]:
protein_name = "trp_cage"  # Change this to your protein of interest

# Initialize Hydra (only needs to be done once per session)
hydra.initialize(config_path="../src/bioemu/config")

cfg = hydra.compose(
    config_name="config",
    overrides=[f"protein_name={protein_name}"]
)
print(f"Using protein: {cfg.protein_name}")

Using protein: trp_cage


/tmp/ipykernel_2153555/1271529333.py:4: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  hydra.initialize(config_path="../src/bioemu/config")
/home/ishan/miniforge3/envs/bioemu/lib/python3.10/site-packages/hydra/_internal/defaults_list.py:251: UserWarning: In 'config': Defaults list is missing `_self_`. See https://hydra.cc/docs/1.2/upgrades/1.0_to_1.1/default_composition_order for more information
  warnings.warn(msg, UserWarning)


## Initialize Model and Load Data

In [3]:
# Initialize trajectory and interpolator
Trajectory = hydra.utils.instantiate(cfg.dataset)
rescaled_protein_trajectory = Trajectory(protein_name=cfg.protein_name, center_scale_traj=True)
rescaled_protein_trajectory = cast(FastFolderTrajectory, rescaled_protein_trajectory)

FastFolderInterpolator = hydra.utils.instantiate(cfg.interpolator)
torch.manual_seed(cfg.seed)
bioemu_interpolator = FastFolderInterpolator(
    dt=cfg.dt,
    path_length=cfg.path_length,
    protein_trajectory=rescaled_protein_trajectory,
)
bioemu_interpolator = cast(Interpolator, bioemu_interpolator)
bioemu_interpolator = bioemu_interpolator.cuda()

Extra atoms in SER: ['OXT']
Removed 1 extra atoms from the topology
Topology has 143 atoms and 278 bonds


## Select and Process Single Frame

In [25]:
# Select a random frame
frame_idx = np.random.randint(0, len(rescaled_protein_trajectory.ground_truth_traj_FAX) - 1)
next_frame_idx = frame_idx + 1

# Get frames
start_frame_FAX = rescaled_protein_trajectory.ground_truth_traj_FAX[frame_idx].unsqueeze(0)
next_frame_FAX = rescaled_protein_trajectory.ground_truth_traj_FAX[next_frame_idx].unsqueeze(0)

# Convert to backbone representation
start_frame_FAbX = bioemu_interpolator.all_atom_to_backbone(start_frame_FAX)
next_frame_FAbX = bioemu_interpolator.all_atom_to_backbone(next_frame_FAX)

# Calculate displacement-based force
force_unit_vec_FAbX = start_frame_FAbX - next_frame_FAbX
force_unit_vec_FAbX /= torch.linalg.vector_norm(force_unit_vec_FAbX, dim=-1, keepdim=True)

# Get model predictions at different noise levels
noise_levels = [0.01, 0.05, 0.1, 0.2, 0.5, 0.99]
predicted_forces = {}

for t in noise_levels:
    score = bioemu_interpolator.get_forces(start_frame_FAbX.cuda(), t=t)
    score /= torch.linalg.vector_norm(score, dim=-1, keepdim=True)
    predicted_forces[t] = score.detach().cpu()

## Visualize Structure and Forces

In [28]:
scale = 2.0

# 1. Get the all-atom coordinates and topology for the frame
coords_all = start_frame_FAX.squeeze().cpu().numpy()  # shape (N_atoms, 3)
topology = rescaled_protein_trajectory.topology  # all-atom mdtraj.Topology

# 2. Get the backbone mask and indices
backbone_mask = rescaled_protein_trajectory.backbone_mask_A.numpy()  # shape (N_atoms,)
backbone_indices = np.where(backbone_mask)[0]  # indices of backbone atoms

# 3. Get the backbone coordinates and forces
coords_backbone = coords_all[backbone_indices]  # shape (N_backbone, 3)
# forces_backbone = force_unit_vec_FAbX.squeeze().cpu().numpy()  # shape (N_backbone, 3)
forces_backbone = predicted_forces[0.05].squeeze().cpu().numpy()

traj = md.Trajectory(coords_all[None, :, :] / 10, topology)

view = nv.show_mdtraj(traj)
view.clear_representations()
view.add_ball_and_stick()

for i, idx in enumerate(backbone_indices):
    start = coords_all[idx]  # in Angstroms
    start_nm = start / 10    # convert to nm for MDTraj
    end_nm = start_nm + (forces_backbone[i] / 10) * scale  # scale force and convert to nm
    view.shape.add_arrow(start_nm * 10, end_nm * 10, [1, 0, 0], 0.2)  # nglview expects Å

view

NGLWidget()

In [ ]:
# Visualize model predictions at different noise levels
for t, forces in predicted_forces.items():
    print(f"\nNoise level: {t}")
    view = visualize_forces(start_frame_FAbX, forces)
    display(view)

## Compare Forces

Calculate cosine similarity between displacement-based forces and model predictions:

In [ ]:
similarities = {}
for t, forces in predicted_forces.items():
    similarity = torch.einsum("ij,ij->i", force_unit_vec_FAbX, forces)
    similarities[t] = similarity.mean().item()

plt.figure(figsize=(10, 5))
plt.plot(list(similarities.keys()), list(similarities.values()), 'o-')
plt.xlabel("Noise Level")
plt.ylabel("Cosine Similarity")
plt.title("Similarity between Displacement and Predicted Forces")
plt.grid(True)
plt.show()

## AMBER Force Comparison

To compare with AMBER forces, you would need to:
1. Save the current frame as a PDB file
2. Run AMBER minimization/energy calculation
3. Load the forces back and compare

This would require AMBER to be installed and properly configured. Would you like me to add this functionality?